# 📝 Assignment: Explainable AI for Text — LIME, SHAP & Integrated Gradients

**Student:** Pushpak V Reddy  
**USN:** 1RUB25CSE0014


**Dataset:** Emotion (dair-ai/emotion) — 20,000 English tweets labelled with **6 emotions**: sadness 😢, joy 😀, love ❤️, anger 😠, fear 😨, surprise 😲
**Model:** `bhadresh-savani/distilbert-base-uncased-emotion` (pre-trained, ~93% accuracy)

![roadmap](attachment:roadmap.png)

### 📋 Instructions
| | |
|---|---|
| ✏️ **Code cells marked `YOUR CODE HERE`** | Write your solution there. Everything else is given — just run it. |
| ✅ **Check cells** (`run_checks(...)`) | Run them after each task. They test your code and show ✅ / ❌. **Do not edit them.** |
| ✍️ **Answer cells** | Double-click and write 2–4 sentences. Use evidence from *your* outputs. |
| 🔑 **Hints** | Click to open — using them costs no marks. |
| 📤 **Submit** | `Runtime → Restart and run all`, then download as `RollNo_Name_XAI.ipynb`. |

<div style="background:#fffbeb;border-left:5px solid #F59E0B;padding:10px 14px;border-radius:6px;color:#111">⚠️ <b>New in this assignment:</b> the demo lab had 2 classes. Here there are <b>6</b>, so every explanation must say <b>which emotion</b> it explains. A word can support <i>fear</i> and argue against <i>joy</i> at the same time.</div>

![multiclass](attachment:multiclass.png)

## ⚙️ Setup (given — just run)
Colab: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# Colab / Jupyter setup. Dataset and model downloads require internet access.
%pip install -q transformers datasets lime shap captum ipywidgets plotly scikit-learn scipy

In [ ]:
import re, time, random, warnings, html as _html
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import plotly.express as px, plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy.stats import spearmanr

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- auto-checker (do not edit) ----------------
CHECKS = {}
def register(qid, *tests): CHECKS[qid] = tests

def _run(qid):
    out = []
    for desc, fn in CHECKS[qid]:
        try:
            ok, err = bool(fn()), ""
        except NameError as e:
            ok, err = False, f"not defined yet — {e}"
        except Exception as e:
            ok, err = False, f"{type(e).__name__}: {str(e)[:90]}"
        out.append((desc, ok, err))
    return out

def run_checks(qid):
    res = _run(qid); n = sum(ok for _, ok, _ in res)
    rows = "".join(f"<tr><td>{'✅' if ok else '❌'}</td><td>{_html.escape(d)}</td>"
                   f"<td style='color:#b91c1c;font-size:12px'>{_html.escape(e)}</td></tr>" for d, ok, e in res)
    col = "#10B981" if n == len(res) else "#F59E0B"
    display(HTML(f"<div style='border:2px solid {col};border-radius:8px;padding:8px 12px;background:#fff;color:#111'>"
                 f"<b style='color:{col}'>{qid}: {n}/{len(res)} checks passed {'🎉' if n == len(res) else ''}</b>"
                 f"<table style='margin-top:4px'>{rows}</table></div>"))

T1 = "i feel so lucky but scared of losing it all"
T2 = "i am so happy and excited about today"
register("Q1",
    ("demo_df has 120 rows", lambda: len(demo_df) == 120),
    ("exactly 20 tweets for each of the 6 emotions", lambda: demo_df.label.value_counts().eq(20).all() and demo_df.label.nunique() == 6),
    ("every tweet has 8–30 words", lambda: demo_df.n_words.between(8, 30).all()),
    ("rows are shuffled (first 12 rows not one class)", lambda: demo_df.label[:12].nunique() > 1),
    ("index reset to 0..119", lambda: list(demo_df.index) == list(range(120))))
register("Q2",
    ("single string → shape (1, 6)", lambda: predict_proba(T1).shape == (1, 6)),
    ("list of 3 strings → shape (3, 6)", lambda: predict_proba([T1, T2, T1]).shape == (3, 6)),
    ("every row sums to 1", lambda: np.allclose(predict_proba([T1, T2]).sum(1), 1, atol=1e-4)),
    ("'i am so happy…' is predicted as joy", lambda: EMOTIONS[int(predict_proba(T2)[0].argmax())] == "joy"))
register("Q3",
    ("returns dict with words, scores, weights, r2", lambda: {"words", "scores", "weights", "r2"} <= set(explain_lime(T1, 1, num_samples=300))),
    ("one score per word", lambda: len(explain_lime(T1, 1, num_samples=300)["scores"]) == len(T1.split())),
    ("some scores are non-zero", lambda: np.abs(explain_lime(T1, 1, num_samples=300)["scores"]).sum() > 0),
    ("same seed → same explanation", lambda: np.allclose(explain_lime(T1, 1, num_samples=300, seed=7)["scores"],
                                                        explain_lime(T1, 1, num_samples=300, seed=7)["scores"])))
register("Q5",
    ("lime_stability returns a number in [0, 1]", lambda: 0 <= lime_stability(T1, 1, 200, seeds=range(3)) <= 1))
register("Q6",
    ("shap_explainer exists", lambda: shap_explainer is not None),
    ("returns dict with sv, tokens, tok_scores, words, scores, base", lambda: {"sv", "tokens", "tok_scores", "words", "scores", "base"} <= set(explain_shap(T1, 1, max_evals=200))),
    ("one score per word", lambda: len(explain_shap(T1, 1, max_evals=200)["scores"]) == len(T1.split())),
    ("additivity: base + Σφ ≈ f(x)  (±0.02)", lambda: (lambda r: abs(r["base"] + r["tok_scores"].sum() - predict_proba(T1)[0, 1]) < 0.02)(explain_shap(T1, 1, max_evals=200))))
register("Q7",
    ("shap_all_classes returns a DataFrame with the 6 emotion columns", lambda: list(shap_all_classes(T1, max_evals=200).columns) == EMOTIONS),
    ("one row per word", lambda: len(shap_all_classes(T1, max_evals=200)) == len(T1.split())))
register("Q8",
    ("global_shap returns a pandas Series", lambda: isinstance(global_shap("joy", n=6), pd.Series)),
    ("Series is sorted from most positive to most negative", lambda: global_shap("joy", n=6).is_monotonic_decreasing))
register("Q9",
    ("returns dict with tokens, tok_scores, words, scores, delta, p_x, p_b", lambda: {"tokens", "tok_scores", "words", "scores", "delta", "p_x", "p_b"} <= set(explain_ig(T1, 1))),
    ("one score per word", lambda: len(explain_ig(T1, 1)["scores"]) == len(T1.split())),
    ("completeness: Σ IG ≈ F(x) − F(x′)  (±0.05, n_steps=100)", lambda: (lambda r: abs(r["tok_scores"].sum() - (r["p_x"] - r["p_b"])) < 0.05)(explain_ig(T1, 1, n_steps=100))),
    ("MASK baseline also works", lambda: len(explain_ig(T1, 1, baseline="MASK")["scores"]) == len(T1.split())))
register("Q11",
    ("compare returns DataFrame with columns word, LIME, SHAP, IG", lambda: list(compare(T1, 1).columns) == ["word", "LIME", "SHAP", "IG"]),
    ("one row per word", lambda: len(compare(T1, 1)) == len(T1.split())))
register("Q12",
    ("aopc returns a float", lambda: isinstance(aopc(T1, np.arange(len(T1.split())), 1), float)),
    ("deleting SHAP-top words hurts more than deleting SHAP-bottom words",
     lambda: (lambda s: aopc(T1, s, 1) > aopc(T1, -s, 1))(explain_shap(T1, 1)["scores"])),
    ("random_aopc returns a float", lambda: isinstance(random_aopc(T1, 1), float)))

def scorecard():
    rows = []
    for q in CHECKS:
        r = _run(q); rows.append((q, sum(ok for _, ok, _ in r), len(r)))
    html = "".join(f"<tr><td><b>{q}</b></td><td>{'🟩' * p}{'⬜' * (t - p)}</td><td>{p}/{t}</td></tr>" for q, p, t in rows)
    display(HTML(f"<table style='color:#111;background:#fff'>{html}</table>"))

print(f"✅ Setup done | device = {device}")

---
# Part A — Data & Model *(15 marks)*

## A.1 Load the dataset (given)

In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", split="test")
EMOTIONS = ds.features["label"].names           # ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
df = ds.to_pandas()
df["emotion"] = df["label"].map(dict(enumerate(EMOTIONS)))
df["n_words"] = df["text"].str.split().str.len()
print(len(df), "tweets | classes:", EMOTIONS)
df.sample(5, random_state=SEED)

### Q1 *(5 marks)* — Explore & build the demo set
Write code that:
1. Draws a **bar chart** of the number of tweets per emotion (Plotly). Is the dataset balanced?
2. Prints the **average number of words** per emotion.
3. Creates **`demo_df`**: exactly **20 tweets per emotion** with **8–30 words**, sampled with `random_state=SEED`, then **shuffled** (`random_state=SEED`) and with the **index reset**. → 120 rows.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Filter with <code>df.n_words.between(8, 30)</code>, sample each class with a list comprehension, join with <code>pd.concat</code>, then <code>.sample(frac=1, random_state=SEED).reset_index(drop=True)</code>.
</details>

In [ ]:
# Q1 — Explore the dataset and create the balanced demo set.
counts = (df.groupby("emotion").size()
          .reindex(EMOTIONS, fill_value=0)
          .rename("tweets").reset_index())
px.bar(counts, x="emotion", y="tweets", category_orders={"emotion": EMOTIONS},
       title="Emotion counts in the original test split",
       labels={"emotion": "Emotion", "tweets": "Number of tweets"},
       text="tweets").update_layout(xaxis_title="Emotion", yaxis_title="Tweets").show()

avg_words = (df.groupby("emotion")["n_words"].mean()
             .reindex(EMOTIONS).rename("average_words").round(2))
print("Average number of whitespace-separated words by emotion:")
display(avg_words.to_frame())

eligible = df.loc[df["n_words"].between(8, 30)].copy()
class_samples = []
for class_id, emotion_name in enumerate(EMOTIONS):
    class_pool = eligible.loc[eligible["label"] == class_id]
    if len(class_pool) < 20:
        raise ValueError(f"Only {len(class_pool)} eligible tweets for {emotion_name}; cannot sample 20.")
    class_samples.append(class_pool.sample(n=20, random_state=SEED))

demo_df = (pd.concat(class_samples, ignore_index=True)
           .sample(frac=1, random_state=SEED)
           .reset_index(drop=True))
print(f"Balanced demo set: {len(demo_df)} tweets")
display(demo_df[["text", "emotion", "n_words"]].head(10))

In [ ]:
run_checks("Q1")

**Q1(d) — Answer:** The original test split is **imbalanced** when the class-count bar chart shows different counts for the six emotions. Accuracy can be misleading because frequent classes contribute more examples and can dominate the overall score; per-class precision, recall, F1 and the confusion matrix provide a more complete view. The `demo_df` sample is deliberately balanced (20 tweets per emotion) to make the explanation experiments easier to compare.

## A.2 Load the model (given)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bhadresh-savani/distilbert-base-uncased-emotion"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
print(model.config.id2label)
assert [model.config.id2label[i] for i in range(6)] == EMOTIONS, "label order mismatch!"
print("✅ model label order matches the dataset")

### Q2 *(6 marks)* — The prediction function every explainer needs
Write **`predict_proba(texts, batch_size=32)`**:
- input: a single string **or** a list/array of strings
- output: a **NumPy array of shape `(n, 6)`** with softmax probabilities
- use `torch.no_grad()`, `padding=True, truncation=True, max_length=128`, and process in batches

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Same pattern as the demo lab: <code>tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)</code> → <code>model(**enc).logits</code> → <code>torch.softmax(..., dim=-1)</code>. Convert with <code>.cpu().numpy()</code> and <code>np.vstack</code> the batches. LIME/SHAP may pass a NumPy array — convert each item with <code>str()</code>.
</details>

In [ ]:
# Q2 — Batched prediction function used by all explainers.
def predict_proba(texts, batch_size=32):
    """Return class probabilities with shape (number_of_texts, 6)."""
    if isinstance(texts, str):
        text_list = [texts]
    elif isinstance(texts, np.ndarray):
        text_list = texts.tolist()
    else:
        text_list = list(texts)

    text_list = [str(text) for text in text_list]
    if batch_size is None or int(batch_size) < 1:
        raise ValueError("batch_size must be a positive integer")
    if not text_list:
        return np.empty((0, len(EMOTIONS)), dtype=np.float32)

    probability_batches = []
    model.eval()
    for start in range(0, len(text_list), int(batch_size)):
        batch = text_list[start:start + int(batch_size)]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=128,
                        return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)
        probability_batches.append(probs.detach().cpu().numpy())

    result = np.vstack(probability_batches).astype(np.float64, copy=False)
    # Normalize very small numerical drift while preserving genuine probabilities.
    row_sums = result.sum(axis=1, keepdims=True)
    result = result / np.clip(row_sums, 1e-12, None)
    return result

In [ ]:
run_checks("Q2")

### Model evaluation on `demo_df` (given)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
P = predict_proba(demo_df.text.tolist())          # (120, 6) — used later!
demo_df["pred"] = P.argmax(1)
cm = confusion_matrix(demo_df.label, demo_df.pred, labels=range(6))
px.imshow(cm, x=EMOTIONS, y=EMOTIONS, text_auto=True, color_continuous_scale="Blues",
          labels=dict(x="predicted", y="true"), title=f"Accuracy {np.mean(demo_df.pred == demo_df.label):.1%}").show()
print(classification_report(demo_df.label, demo_df.pred, target_names=EMOTIONS, digits=2))

In [ ]:
# Data-dependent answer for Q2(b): report the largest off-diagonal confusion.
confusion_offdiag = cm.copy()
np.fill_diagonal(confusion_offdiag, 0)
if confusion_offdiag.max() > 0:
    true_id, pred_id = np.unravel_index(np.argmax(confusion_offdiag), confusion_offdiag.shape)
    top_confusion_pair = (EMOTIONS[int(true_id)], EMOTIONS[int(pred_id)])
    print(f"Most frequent directed confusion: true {top_confusion_pair[0]!r} → predicted {top_confusion_pair[1]!r} "
          f"({int(confusion_offdiag[true_id, pred_id])} tweets).")
else:
    top_confusion_pair = None
    print("No off-diagonal confusions were observed in this demo set.")

**Q2(b) — Answer:** The most-confused pair is the true/predicted emotion pair printed by the confusion-matrix summary immediately above this response. These classes can be confused because tweets are short, context is limited, and emotion categories can share words or sentiment cues; the direction of confusion also matters (true emotion versus predicted emotion).

### 🧰 Helpers (given — run once)
All explainers below must return **one score per whitespace word** (`text.split()`), measured as contribution to **P(chosen emotion)**. These helpers draw and align them.

In [ ]:
EMO_EMOJI = dict(sadness="😢", joy="😀", love="❤️", anger="😠", fear="😨", surprise="😲")

def highlight(words, scores, title="", note=""):
    scores = np.asarray(scores, float); m = np.abs(scores).max() or 1.0
    spans = "".join(f"<span title='{s:+.4f}' style='background:rgba({'16,185,129' if s > 0 else '239,68,68'},"
                    f"{min(1, abs(s) / m) * .85:.2f});color:#111;padding:2px 4px;margin:2px;border-radius:5px;"
                    f"display:inline-block'>{_html.escape(w)}</span>" for w, s in zip(words, scores))
    return (f"<div style='border:1px solid #e5e7eb;border-radius:10px;padding:8px 12px;margin:5px 0;background:#fff;"
            f"color:#111;line-height:1.9'><b>{title}</b> <span style='color:#666;font-size:12px'>{note}</span><br>{spans}</div>")

def prob_bars(p):
    fig = px.bar(x=[f"{EMO_EMOJI[e]} {e}" for e in EMOTIONS], y=p, range_y=[0, 1], labels={"x": "", "y": "probability"})
    fig.update_layout(height=250, margin=dict(t=20, b=20)); return fig

def align_to_words(text, tokens, scores):
    """Sum sub-word token scores onto whitespace words of `text`."""
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    low, cur, out = text.lower(), 0, np.zeros(len(spans))
    for tok, s in zip(tokens, scores):
        t = str(tok).replace("##", "").strip().lower()
        if not t or t in {"[cls]", "[sep]", "[pad]", "[unk]", "[mask]"}: continue
        pos = low.find(t, cur)
        if pos == -1: continue
        cur = pos + len(t)
        for i, (a, b) in enumerate(spans):
            if a <= pos < b: out[i] += s; break
    return text.split(), out

CACHE = {}
def cached(method, text, **kw):
    """cached("LIME"|"SHAP"|"IG", text, label=..., ...) → runs explain_<method> once, adds 'time'."""
    key = (method, text, tuple(sorted(kw.items())))
    if key not in CACHE:
        t0 = time.time(); r = globals()[f"explain_{method.lower()}"](text, **kw)
        r["time"] = time.time() - t0; CACHE[key] = r
    return CACHE[key]

def tweet_picker(default=0):
    opts = [(f"#{i} [{EMOTIONS[l]}] {t[:65]}", i) for i, (t, l) in enumerate(zip(demo_df.text, demo_df.label))]
    dd = widgets.Dropdown(options=opts, value=default, description="Tweet:", layout=widgets.Layout(width="95%"))
    ta = widgets.Textarea(value=demo_df.text[default], layout=widgets.Layout(width="95%", height="60px"))
    dd.observe(lambda ch: setattr(ta, "value", demo_df.text[ch["new"]]), names="value")
    return dd, ta
print("✅ helpers ready")

---
# Part B — LIME *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> LIME removes random words, queries the model, and fits a weighted linear model. With 6 classes, pass <code>labels=(label,)</code> to choose <b>which</b> emotion to explain.</div>

### Q3 *(8 marks)* — Implement `explain_lime`
```python
explain_lime(text, label, num_features=10, num_samples=1000, seed=SEED) -> dict
```
Return a dict with:
- `exp` — the LIME explanation object
- `weights` — `{word: weight}` from `exp.as_list(label=label)`
- `words` — `text.split()`
- `scores` — NumPy array, one score per word (0 if the word isn't in `weights`)
- `r2` — the local surrogate's fit score (`exp.score`)

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Create <code>LimeTextExplainer(class_names=EMOTIONS, random_state=seed)</code> <b>inside</b> the function so the seed is respected. LIME splits on <code>\W+</code>, so a word like <code>don't</code> becomes <code>don</code> + <code>t</code>: sum <code>weights.get(piece, 0)</code> over <code>re.split(r'\W+', word)</code>.
</details>

In [ ]:
# Q3 — LIME explanation for one selected emotion.
from lime.lime_text import LimeTextExplainer

def explain_lime(text, label, num_features=10, num_samples=1000, seed=SEED):
    text = str(text)
    label = int(label)
    words = text.split()
    if not words:
        return {"exp": None, "weights": {}, "words": [], "scores": np.array([], dtype=float), "r2": 0.0}

    # Build the explainer inside the function so every seed is reproducible.
    explainer = LimeTextExplainer(class_names=EMOTIONS, random_state=int(seed))
    exp = explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_proba,
        labels=(label,),
        num_features=max(1, min(int(num_features), len(words))),
        num_samples=max(10, int(num_samples)),
    )
    weights = {str(word): float(weight) for word, weight in exp.as_list(label=label)}

    # LIME splits punctuation (e.g. "don't" → "don", "t"). Sum component weights
    # for each original whitespace word, as requested by the assignment.
    lower_weights = {}
    for feature, weight in weights.items():
        lower_weights[feature.lower()] = weight
    scores = []
    for word in words:
        pieces = [piece for piece in re.split(r"\W+", word) if piece]
        score = 0.0
        for piece in pieces:
            score += weights.get(piece, lower_weights.get(piece.lower(), 0.0))
        scores.append(score)

    return {"exp": exp, "weights": weights, "words": words,
            "scores": np.asarray(scores, dtype=float), "r2": float(getattr(exp, "score", 0.0))}

In [ ]:
run_checks("Q3")

### Q4 *(6 marks)* — Explaining an ambiguous tweet
1. Using `P` (probabilities for `demo_df`), find **`amb_idx`**: the tweet whose **top-1 and top-2 probabilities are closest** (the most ambiguous tweet).
2. Show its probabilities with `prob_bars(...)`.
3. Run `explain_lime` **twice** — for the top-1 emotion and the top-2 emotion — and display both with `highlight(...)`.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>S = np.sort(P, axis=1)</code> → gap = <code>S[:, -1] - S[:, -2]</code> → <code>amb_idx = int(np.argmin(gap))</code>. Top-2 labels: <code>np.argsort(-P[amb_idx])[:2]</code>.
</details>

In [ ]:
# Q4 — Find the most ambiguous tweet and explain the top two candidate emotions.
prob_sorted = np.sort(np.asarray(P), axis=1)
ambiguity_gaps = prob_sorted[:, -1] - prob_sorted[:, -2]
amb_idx = int(np.argmin(ambiguity_gaps))
amb_text = str(demo_df.iloc[amb_idx]["text"])
amb_probs = np.asarray(P[amb_idx], dtype=float)
amb_labels = np.argsort(-amb_probs)[:2].astype(int).tolist()
amb_top_label, amb_runnerup_label = amb_labels

print(f"Ambiguous tweet index: {amb_idx} | top-two probability gap: {ambiguity_gaps[amb_idx]:.4f}")
print(amb_text)
prob_bars(amb_probs).show()

lime_top = explain_lime(amb_text, amb_top_label, num_features=len(amb_text.split()), num_samples=1000, seed=SEED)
lime_runnerup = explain_lime(amb_text, amb_runnerup_label, num_features=len(amb_text.split()), num_samples=1000, seed=SEED)
display(HTML(highlight(lime_top["words"], lime_top["scores"],
                       f"LIME → {EMOTIONS[amb_top_label]}", f"local R²={lime_top['r2']:.3f}")))
display(HTML(highlight(lime_runnerup["words"], lime_runnerup["scores"],
                       f"LIME → {EMOTIONS[amb_runnerup_label]}", f"local R²={lime_runnerup['r2']:.3f}")))

# A concrete example, when one exists, of a word pushing the two labels in opposite directions.
opposite_positions = [i for i, (a, b) in enumerate(zip(lime_top["scores"], lime_runnerup["scores"])) if a * b < 0]
if opposite_positions:
    best_i = max(opposite_positions, key=lambda i: abs(lime_top["scores"][i]) + abs(lime_runnerup["scores"][i]))
    print(f"Opposite-sign example: {lime_top['words'][best_i]!r}: "
          f"{lime_top['scores'][best_i]:+.4f} toward {EMOTIONS[amb_top_label]}, "
          f"{lime_runnerup['scores'][best_i]:+.4f} toward {EMOTIONS[amb_runnerup_label]}.")
else:
    print("No word has opposite-sign LIME scores for these two labels in this run; compare the two highlights directly.")

**Q4(d) — Answer:** The Q4 output prints an example, when one exists, where a word has a positive LIME score for the top emotion and a negative score for the runner-up. This shows that in a multiclass problem, a word's contribution is **relative to the chosen emotion**—there is no single class-independent meaning of “important.” If no opposite-sign example is found, compare the two displayed highlights and report the strongest contrast.

### Q5 *(6 marks)* — How stable is LIME?
Write **`lime_stability(text, label, num_samples, seeds=range(5), k=5)`** that runs LIME once per seed, takes the **top-k words by |weight|**, and returns the **mean pairwise Jaccard similarity** of those top-k sets (1.0 = identical every time).

Then plot stability for `num_samples` in `[100, 300, 1000, 2000]` on `amb_text` (top-1 emotion).

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Jaccard(A, B) = |A ∩ B| / |A ∪ B|. Use <code>itertools.combinations(sets, 2)</code> for the pairs. Top-k: <code>sorted(r['weights'], key=lambda w: -abs(r['weights'][w]))[:k]</code>.
</details>

In [ ]:
# Q5 — LIME top-word stability as the perturbation sample count grows.
from itertools import combinations

def lime_stability(text, label, num_samples, seeds=range(5), k=5):
    top_sets = []
    for seed in seeds:
        result = explain_lime(text, label, num_features=max(1, len(str(text).split())),
                              num_samples=num_samples, seed=int(seed))
        ranked = sorted(result["weights"], key=lambda word: -abs(result["weights"][word]))[:max(1, int(k))]
        top_sets.append(set(ranked))
    pair_scores = []
    for first, second in combinations(top_sets, 2):
        union = first | second
        pair_scores.append(len(first & second) / len(union) if union else 1.0)
    return float(np.mean(pair_scores)) if pair_scores else 1.0

stability_rows = []
for sample_count in [100, 300, 1000, 2000]:
    score = lime_stability(amb_text, amb_top_label, sample_count, seeds=range(5), k=5)
    stability_rows.append({"num_samples": sample_count, "stability": score})
stability_df = pd.DataFrame(stability_rows)
display(stability_df.round(3))
px.line(stability_df, x="num_samples", y="stability", markers=True,
       range_y=[0, 1], title=f"LIME top-5 stability for {EMOTIONS[amb_top_label]}",
       labels={"num_samples": "LIME perturbation samples", "stability": "Mean pairwise Jaccard"}).show()

In [ ]:
run_checks("Q5")

**Q5(b) — Answer:** The stability curve shows how consistently LIME selects the same top-five words across random seeds. I would start with **1,000 samples** as a practical balance between runtime and repeatability, then move to 2,000 if the score at 1,000 is still low or the top words change noticeably. The best choice is the smallest sample count at which the curve has largely levelled off.

---
# Part C — SHAP *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> SHAP values are fair (Shapley) shares of the prediction. <b>Additivity:</b> base value + Σ SHAP values = model output for the explained class.</div>

### Q6 *(8 marks)* — Implement `explain_shap`
1. Create a global **`shap_explainer`** = `shap.Explainer(predict_proba, shap.maskers.Text(tokenizer), output_names=EMOTIONS)`.
2. Write **`explain_shap(text, label, max_evals=500)`** returning a dict with:
   - `sv` — the SHAP Explanation (`shap_explainer([text], max_evals=..., silent=True)`)
   - `tokens` — `list(sv.data[0])`
   - `tok_scores` — SHAP values for column `label` (NumPy array)
   - `words`, `scores` — from `align_to_words(text, tokens, tok_scores)`
   - `base` — base value for `label`
3. Show `shap.plots.text(...)` for `amb_text` and its top-1 emotion.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>sv.values[0]</code> has shape <code>(n_tokens, 6)</code> → take <code>[:, label]</code>. <code>sv.base_values[0][label]</code> is the base. Plot: <code>shap.plots.text(r['sv'][0, :, label])</code>.
</details>

In [ ]:
# Q6 — SHAP text explanation (all six output classes are returned in one call).
import shap

shap_explainer = shap.Explainer(predict_proba, shap.maskers.Text(tokenizer), output_names=EMOTIONS)
_SHAP_RAW_CACHE = {}

def _get_shap_values(text, max_evals=500):
    key = (str(text), int(max_evals))
    if key not in _SHAP_RAW_CACHE:
        try:
            raw = shap_explainer([str(text)], max_evals=int(max_evals), silent=True)
        except TypeError:
            # Compatibility with SHAP versions whose call signature lacks `silent`.
            raw = shap_explainer([str(text)], max_evals=int(max_evals))
        _SHAP_RAW_CACHE[key] = raw
    return _SHAP_RAW_CACHE[key]

def explain_shap(text, label, max_evals=500):
    text = str(text)
    label = int(label)
    sv = _get_shap_values(text, max_evals=max_evals)
    tokens = [str(token) for token in sv.data[0]]
    values = np.asarray(sv.values[0])
    if values.ndim == 1:
        # Some SHAP builds squeeze a single output; the six-class model should not.
        values = values[:, None]
    if label < 0 or label >= values.shape[-1]:
        raise ValueError(f"label must be between 0 and {values.shape[-1]-1}")
    tok_scores = np.asarray(values[:, label], dtype=float)
    words, scores = align_to_words(text, tokens, tok_scores)
    bases = np.asarray(sv.base_values[0]).reshape(-1)
    base = float(bases[label]) if len(bases) > label else float(bases[0])
    return {"sv": sv, "tokens": tokens, "tok_scores": tok_scores,
            "words": words, "scores": np.asarray(scores, dtype=float), "base": base}

shap_demo = explain_shap(amb_text, amb_top_label, max_evals=500)
try:
    display(shap.plots.text(shap_demo["sv"][0, :, amb_top_label]))
except Exception as exc:
    # The explanation values remain available even if this SHAP version cannot render rich text.
    print(f"SHAP text rendering was unavailable ({type(exc).__name__}: {exc}); showing word scores instead.")
    display(pd.DataFrame({"word": shap_demo["words"], "SHAP": shap_demo["scores"]}))

In [ ]:
run_checks("Q6")

### Q7 *(6 marks)* — One tweet, six explanations
Write **`shap_all_classes(text, max_evals=500)`** that returns a DataFrame: **one row per word** (index like `"word·i"`), **one column per emotion** (`EMOTIONS` order), values = word-level SHAP scores. Call SHAP **only once** (one call already returns all 6 classes).

Draw it as a heat-map (`px.imshow`, diverging colours, midpoint 0) for `amb_text`.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Loop <code>for c in range(6)</code> over <code>np.array(sv.values[0])[:, c]</code> and align each column with <code>align_to_words</code>.
</details>

In [ ]:
# Q7 — Align one SHAP call into a word-by-emotion matrix.
def shap_all_classes(text, max_evals=500):
    text = str(text)
    sv = _get_shap_values(text, max_evals=max_evals)  # one explainer call for all six classes
    tokens = [str(token) for token in sv.data[0]]
    values = np.asarray(sv.values[0])
    if values.ndim == 1:
        values = values[:, None]
    words = text.split()
    class_scores = {}
    for class_id, emotion_name in enumerate(EMOTIONS):
        _, word_scores = align_to_words(text, tokens, values[:, class_id])
        class_scores[emotion_name] = np.asarray(word_scores, dtype=float)
    labels = [f"{word}·{i}" for i, word in enumerate(words)]
    return pd.DataFrame(class_scores, index=labels, columns=EMOTIONS)

shap_class_df = shap_all_classes(amb_text, max_evals=500)
px.imshow(shap_class_df, aspect="auto", color_continuous_scale="RdBu",
          color_continuous_midpoint=0,
          labels={"x": "Emotion", "y": "Word occurrence", "color": "SHAP"},
          title="Word-level SHAP contributions across all six emotions").update_layout(height=480).show()
display(shap_class_df.round(4))

In [ ]:
run_checks("Q7")

### Q8 *(6 marks)* — From local to global
Write **`global_shap(emotion, n=40)`**: explain the first `n` tweets of `demo_df` toward `emotion` (use `cached("SHAP", text, label=...)`), clean each word (`re.sub(r"\W", "", w.lower())`), drop empty words, and return a **Series** of the **mean SHAP per word** for words seen **≥ 2 times**, sorted **descending**.

Then run the given 🎮 explorer below.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Collect <code>(word, score)</code> pairs into a DataFrame → <code>groupby('word')['shap'].agg(['mean', 'count'])</code> → filter <code>count >= 2</code> → <code>['mean'].sort_values(ascending=False)</code>.
</details>

In [ ]:
# Q8 — Aggregate local SHAP contributions into global word-level mean scores.
def global_shap(emotion, n=40):
    if emotion not in EMOTIONS:
        raise ValueError(f"emotion must be one of {EMOTIONS}")
    label = EMOTIONS.index(emotion)
    records = []
    subset = demo_df.iloc[:max(0, min(int(n), len(demo_df)))]
    for text in subset["text"].astype(str):
        result = cached("SHAP", text, label=label, max_evals=200)
        for word, score in zip(result["words"], result["scores"]):
            clean = re.sub(r"\W", "", word.lower())
            if clean:
                records.append((clean, float(score)))
    if not records:
        return pd.Series(dtype=float, name="mean_shap")
    grouped = (pd.DataFrame(records, columns=["word", "shap"])
               .groupby("word")["shap"].agg(["mean", "count"]))
    result = grouped.loc[grouped["count"] >= 2, "mean"].sort_values(ascending=False)
    result.name = "mean_shap"
    return result

In [ ]:
run_checks("Q8")

In [ ]:
# 🎮 Global explorer (given) — uses YOUR global_shap
emo = widgets.Dropdown(options=EMOTIONS, value="anger", description="emotion")
nn = widgets.IntSlider(40, 10, 120, step=10, description="# tweets")
b, o = widgets.Button(description="Explore ▶", button_style="info"), widgets.Output()
def _go(_):
    with o:
        clear_output(wait=True); print("⏳ explaining…")
        s = global_shap(emo.value, nn.value); top = pd.concat([s.head(10), s.tail(10)]).sort_values()
        clear_output(wait=True)
        px.bar(x=top.values, y=top.index, orientation="h", color=top.values, color_continuous_scale="RdYlGn",
               color_continuous_midpoint=0, labels={"x": f"mean SHAP toward {emo.value}", "y": ""},
               title=f"{EMO_EMOJI[emo.value]} Words that push toward / away from {emo.value}").update_layout(height=520).show()
b.on_click(_go); display(widgets.HBox([emo, nn, b]), o)

In [ ]:
# Q8(b) — Data-derived examples for two emotions (first 40 demo tweets).
q8_global_examples = {}
for emotion_name in ["joy", "anger"]:
    word_means = global_shap(emotion_name, n=40)
    q8_global_examples[emotion_name] = word_means
    print(f"{emotion_name.title()} — top 3 words pushing toward this emotion:")
    if len(word_means):
        display(word_means.head(3).rename("mean_SHAP").to_frame().round(4))
    else:
        print("No words occurred at least twice in the selected sample.")

**Q8(b) — Answer:** The code immediately above prints the top three positive-mean SHAP words for **joy** and **anger**, aggregated over the first 40 demo tweets. They are sensible if the words fit the target emotion in context, but obvious keywords can indicate shortcut learning. A model that over-relies on a few words may fail on negation, sarcasm, unfamiliar phrasing, or a changed data distribution; SHAP describes the model's behaviour, not whether that behaviour is correct.

---
# Part D — Integrated Gradients *(20 marks)*

<div style="background:#ecfdf5;border-left:5px solid #10B981;padding:10px 14px;border-radius:6px;color:#111">💡 <b>Recall:</b> IGᵢ = (xᵢ − x′ᵢ) × ∫₀¹ ∂F(x′ + α(x − x′))/∂xᵢ dα. <b>Completeness:</b> Σ IG = F(x) − F(x′). We attribute at the <b>word-embedding layer</b> with Captum's <code>LayerIntegratedGradients</code>.</div>

### Q9 *(12 marks)* — Implement `explain_ig`
1. `forward_probs(input_ids, attention_mask)` → softmax probabilities `(batch, 6)`
2. `lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)`
3. `make_baseline(ids, kind="PAD")` → same shape as `ids`, every token replaced by `[PAD]` (or `[MASK]` if `kind="MASK"`), **except** the first `[CLS]` and last `[SEP]`
4. `explain_ig(text, label, n_steps=50, baseline="PAD")` → dict with `tokens, tok_scores, words, scores, delta, p_x, p_b`
   - `tok_scores`: attributions **summed over the embedding dimension**
   - `delta`: |convergence delta| as a float; `p_x`, `p_b`: P(label) for input and baseline

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>lig.attribute(inputs=ids, baselines=base, additional_forward_args=(mask,), target=label, n_steps=n_steps, internal_batch_size=25, return_convergence_delta=True)</code> returns <code>(attrs, delta)</code>; <code>attrs</code> has shape <code>(1, seq_len, 768)</code>. Token strings: <code>tokenizer.convert_ids_to_tokens(ids[0].tolist())</code>. IDs: <code>tokenizer.pad_token_id</code>, <code>tokenizer.mask_token_id</code>.
</details>

In [ ]:
# Q9 — Integrated Gradients on the word-embedding layer.
from captum.attr import LayerIntegratedGradients

def forward_probs(input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return torch.softmax(outputs.logits, dim=-1)

lig = LayerIntegratedGradients(forward_probs, model.distilbert.embeddings.word_embeddings)

def make_baseline(ids, kind="PAD"):
    kind = str(kind).upper()
    if kind == "PAD":
        replacement_id = tokenizer.pad_token_id
    elif kind == "MASK":
        replacement_id = tokenizer.mask_token_id
    else:
        raise ValueError("baseline must be 'PAD' or 'MASK'")
    if replacement_id is None:
        raise ValueError(f"Tokenizer does not define a token ID for {kind}")
    baseline_ids = torch.full_like(ids, fill_value=int(replacement_id))
    # Preserve the sentence boundary tokens (normally [CLS] and [SEP]).
    if ids.shape[1] >= 1:
        baseline_ids[:, 0] = ids[:, 0]
    if ids.shape[1] >= 2:
        baseline_ids[:, -1] = ids[:, -1]
    return baseline_ids

def explain_ig(text, label, n_steps=50, baseline="PAD"):
    text = str(text)
    label = int(label)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=False)
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device)
    base_ids = make_baseline(ids, kind=baseline)

    with torch.no_grad():
        p_x = float(forward_probs(ids, mask)[0, label].detach().cpu().item())
        p_b = float(forward_probs(base_ids, mask)[0, label].detach().cpu().item())

    attrs, convergence_delta = lig.attribute(
        inputs=ids,
        baselines=base_ids,
        additional_forward_args=(mask,),
        target=label,
        n_steps=max(1, int(n_steps)),
        internal_batch_size=25,
        return_convergence_delta=True,
    )
    tok_scores = attrs.sum(dim=-1).squeeze(0).detach().cpu().numpy().astype(float)
    tokens = tokenizer.convert_ids_to_tokens(ids[0].detach().cpu().tolist())
    words, scores = align_to_words(text, tokens, tok_scores)
    delta_value = float(torch.as_tensor(convergence_delta).detach().abs().max().cpu().item())
    return {"tokens": tokens, "tok_scores": tok_scores, "words": words,
            "scores": np.asarray(scores, dtype=float), "delta": delta_value,
            "p_x": p_x, "p_b": p_b}

In [ ]:
run_checks("Q9")

### Q10 *(8 marks)* — Two IG experiments
**(a)** For `amb_text`, plot **|δ| vs `n_steps`** for `[5, 10, 25, 50, 100]` (log-log axes).
**(b)** For the first **10 tweets** of `demo_df` (each explained for its predicted emotion), compute the **Spearman correlation** between word scores using the **PAD** and the **MASK** baseline. Print the mean and draw a histogram/strip plot.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>spearmanr(a, b)[0]</code> gives ρ. Use <code>np.nanmean</code> in case a score vector is constant.
</details>

In [ ]:
# Q10(a) — convergence delta versus integration steps; Q10(b) — PAD/MASK sensitivity.
step_rows = []
for steps in [5, 10, 25, 50, 100]:
    result = cached("IG", amb_text, label=amb_top_label, n_steps=steps, baseline="PAD")
    step_rows.append({"n_steps": steps, "abs_delta": max(float(result["delta"]), 1e-10)})
step_df = pd.DataFrame(step_rows)
display(step_df)
px.line(step_df, x="n_steps", y="abs_delta", markers=True, log_x=True, log_y=True,
       title="Integrated Gradients convergence versus n_steps",
       labels={"n_steps": "Integration steps (log scale)", "abs_delta": "|convergence delta| (log scale)"}).show()

correlations = []
for i, row in demo_df.iloc[:10].iterrows():
    text = str(row["text"])
    label = int(row["pred"])
    pad_result = cached("IG", text, label=label, n_steps=50, baseline="PAD")
    mask_result = cached("IG", text, label=label, n_steps=50, baseline="MASK")
    rho = float(spearmanr(pad_result["scores"], mask_result["scores"])[0])
    correlations.append({"tweet_index": int(i), "emotion": EMOTIONS[label], "spearman_rho": rho})

ig_baseline_df = pd.DataFrame(correlations)
mean_rho = float(np.nanmean(ig_baseline_df["spearman_rho"].to_numpy()))
print(f"Mean PAD-versus-MASK word-score Spearman correlation over the first 10 tweets: {mean_rho:.3f}")
display(ig_baseline_df.round(3))
px.histogram(ig_baseline_df, x="spearman_rho", nbins=8,
             title="Agreement between PAD and MASK IG baselines",
             labels={"spearman_rho": "Spearman correlation across word scores"}).show()

**Q10(c) — Answer:** Use the log-log convergence plot to choose the smallest step count after which the absolute delta stops improving materially; 50 or 100 steps may be reasonable if the curve has flattened, but the plotted values should decide. PAD and MASK can yield different word scores because they define different reference inputs. This sensitivity is a weakness when conclusions depend on an arbitrary baseline, but also a feature because the baseline makes the comparison being asked explicit.

---
# Part E — Comparison & Faithfulness *(20 marks)*

### 🎮 Your explainer dashboard (given — works once Q3, Q6, Q9 pass)
Pick any tweet (or type your own), a method and an emotion.

In [ ]:
dd, ta = tweet_picker()
meth = widgets.ToggleButtons(options=["LIME", "SHAP", "IG"], description="method")
emo2 = widgets.Dropdown(options=[("predicted", -1)] + [(e, i) for i, e in enumerate(EMOTIONS)], description="explain")
b, o = widgets.Button(description="Explain ▶", button_style="success"), widgets.Output()
def _exp(_):
    with o:
        clear_output(wait=True); print("⏳ working…")
        p = predict_proba(ta.value)[0]; lab = int(p.argmax()) if emo2.value == -1 else emo2.value
        kw = {"num_features": len(ta.value.split())} if meth.value == "LIME" else {}
        r = cached(meth.value, ta.value, label=lab, **kw)
        clear_output(wait=True); prob_bars(p).show()
        display(HTML(highlight(r["words"], r["scores"], f"{meth.value} → {EMO_EMOJI[EMOTIONS[lab]]} {EMOTIONS[lab]}", f"{r['time']:.1f}s")))
b.on_click(_exp); display(dd, ta, widgets.HBox([meth, emo2]), b, o)

![methods](attachment:methods.png)

### Q11 *(8 marks)* — Head-to-head comparison
Write **`compare(text, label)`** returning a DataFrame with columns **`word, LIME, SHAP, IG`** (one row per word). Use `cached(...)`; for LIME pass `num_features=len(text.split())` so every word gets a score.

Then for `amb_text` (top-1 emotion):
1. Normalise each method's column by its max |score| and draw a **grouped bar chart** of the 10 most important words.
2. Show the **Spearman correlation matrix** between the three methods (heat-map).
3. Print the **top-3 words** of each method.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

<code>C[['LIME','SHAP','IG']].corr(method='spearman')</code>. Most important words: <code>norm.abs().max(axis=1).nlargest(10).index</code>.
</details>

In [ ]:
# Q11 — Compare LIME, SHAP, and IG on the same text and target class.
def compare(text, label):
    text = str(text)
    label = int(label)
    words = text.split()
    r_lime = cached("LIME", text, label=label,
                    num_features=max(1, len(words)), num_samples=1000, seed=SEED)
    r_shap = cached("SHAP", text, label=label, max_evals=300)
    r_ig = cached("IG", text, label=label, n_steps=50, baseline="PAD")
    # Defensive alignment: all three explainers are required to return one score per word.
    for method_name, result in [("LIME", r_lime), ("SHAP", r_shap), ("IG", r_ig)]:
        if len(result["scores"]) != len(words):
            raise ValueError(f"{method_name} returned {len(result['scores'])} scores for {len(words)} words")
    return pd.DataFrame({"word": words, "LIME": r_lime["scores"],
                         "SHAP": r_shap["scores"], "IG": r_ig["scores"]})

comparison_df = compare(amb_text, amb_top_label)
display(comparison_df.round(4))
score_cols = ["LIME", "SHAP", "IG"]
raw_scores = comparison_df[score_cols].astype(float)
normalizers = raw_scores.abs().max(axis=0).replace(0, 1.0)
normalized_scores = raw_scores / normalizers
importance = normalized_scores.abs().max(axis=1)
top_positions = np.argsort(importance.to_numpy())[-min(10, len(comparison_df)):]
top_comparison = normalized_scores.iloc[top_positions].copy()
top_words = comparison_df.iloc[top_positions]["word"].tolist()
top_comparison.insert(0, "word_label", [f"{word} · occurrence {pos+1}" for pos, word in zip(top_positions, top_words)])
plot_long = top_comparison.melt(id_vars="word_label", value_vars=score_cols,
                                var_name="method", value_name="normalized_score")
px.bar(plot_long, x="word_label", y="normalized_score", color="method", barmode="group",
       title=f"Normalized word contributions for {EMOTIONS[amb_top_label]}",
       labels={"word_label": "Word", "normalized_score": "Score / max |score|"}).update_layout(xaxis_tickangle=-35).show()

corr_matrix = comparison_df[score_cols].corr(method="spearman")
px.imshow(corr_matrix, text_auto=".2f", zmin=-1, zmax=1,
          color_continuous_scale="RdBu", color_continuous_midpoint=0,
          title="Spearman correlation between explanation methods").show()
for method_name in score_cols:
    order = np.argsort(-np.abs(comparison_df[method_name].to_numpy()))[:3]
    print(f"Top 3 {method_name} words by absolute contribution:",
          [(comparison_df.iloc[j]["word"], round(float(comparison_df.iloc[j][method_name]), 4)) for j in order])

In [ ]:
run_checks("Q11")

![aopc](attachment:aopc.png)

### Q12 *(12 marks)* — Which explanation is most faithful?
1. **`aopc(text, scores, label, k_max=3)`** → delete the top-k words (highest score = most supportive of `label`) for k = 1…k_max, return the **mean drop** in P(label) as a `float`.
2. **`random_aopc(text, label, k_max=3, n=5, seed=SEED)`** → same but with random word orders, averaged over `n` runs.
3. **Benchmark** on the first **15 tweets** of `demo_df` (explain each for its predicted emotion): a DataFrame `bench` with columns `tweet, method, AOPC, seconds` (methods: LIME, SHAP, IG, Random). Show the **mean table** and a **box plot** of AOPC.

<details><summary>🔑 <b>Hint</b> (click to open)</summary>

Order: <code>np.argsort(-np.asarray(scores))</code>. Build all deleted versions first, then call <code>predict_proba</code> once on the list. Use <code>min(k_max, len(words) - 1)</code> so a tweet never becomes empty. Random order: pass <code>rng.random(n_words)</code> as fake scores to <code>aopc</code>.
</details>

In [ ]:
# Q12 — Deletion-based faithfulness with random-order baseline.
# Clear prior explainer caches so reported generation times are measured afresh.
CACHE.clear()
_SHAP_RAW_CACHE.clear()
def aopc(text, scores, label, k_max=3):
    words = str(text).split()
    if len(words) <= 1:
        return 0.0
    values = np.asarray(scores, dtype=float).reshape(-1)
    if len(values) != len(words):
        raise ValueError(f"Expected one score per whitespace word ({len(words)}), got {len(values)}")
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    max_k = min(max(1, int(k_max)), len(words) - 1)
    ranked_positions = np.argsort(-values)[:max_k]  # most supportive words first
    original_probability = float(predict_proba(str(text))[0, int(label)])
    perturbed_texts = []
    for k in range(1, max_k + 1):
        removed = set(int(idx) for idx in ranked_positions[:k])
        remaining = [word for idx, word in enumerate(words) if idx not in removed]
        perturbed_texts.append(" ".join(remaining) if remaining else " ")
    changed_probabilities = predict_proba(perturbed_texts)[:, int(label)]
    return float(np.mean(original_probability - changed_probabilities))

def random_aopc(text, label, k_max=3, n=5, seed=SEED):
    words = str(text).split()
    if len(words) <= 1:
        return 0.0
    rng = np.random.default_rng(int(seed))
    values = []
    for _ in range(max(1, int(n))):
        random_scores = rng.random(len(words))
        values.append(aopc(text, random_scores, label, k_max=k_max))
    return float(np.mean(values))

benchmark_rows = []
for row_index, row in demo_df.iloc[:15].iterrows():
    text = str(row["text"])
    label = int(row["pred"])
    words = text.split()
    # `seconds` records attribution/ranking generation time, not the shared deletion evaluation.
    method_specs = [
        ("LIME", lambda: cached("LIME", text, label=label, num_features=max(1, len(words)), num_samples=500, seed=SEED)),
        ("SHAP", lambda: cached("SHAP", text, label=label, max_evals=200)),
        ("IG", lambda: cached("IG", text, label=label, n_steps=50, baseline="PAD")),
    ]
    for method_name, get_result in method_specs:
        result = get_result()
        explanation_seconds = float(result.get("time", 0.0))
        method_aopc = aopc(text, result["scores"], label, k_max=3)
        benchmark_rows.append({"tweet": int(row_index), "method": method_name,
                               "AOPC": method_aopc, "seconds": explanation_seconds})

    rng = np.random.default_rng(SEED + int(row_index))
    t0 = time.perf_counter()
    random_scores = rng.random(len(words))
    random_generation_seconds = time.perf_counter() - t0
    random_value = random_aopc(text, label, k_max=3, n=5, seed=SEED + int(row_index))
    benchmark_rows.append({"tweet": int(row_index), "method": "Random",
                           "AOPC": random_value, "seconds": random_generation_seconds})

bench = pd.DataFrame(benchmark_rows, columns=["tweet", "method", "AOPC", "seconds"])
bench_summary = (bench.groupby("method", as_index=False)
                 .agg(mean_AOPC=("AOPC", "mean"), mean_seconds=("seconds", "mean"))
                 .sort_values("mean_AOPC", ascending=False))
print("Mean benchmark results (AOPC: larger is a larger probability drop after deleting supportive words):")
display(bench_summary.round(4))
print("Faithfulness ranking, best to worst:", " > ".join(bench_summary["method"].tolist()))
print("Attribution-generation speed ranking, fastest to slowest:",
      " > ".join(bench_summary.sort_values("mean_seconds")["method"].tolist()))
px.box(bench, x="method", y="AOPC", points="all",
      title="AOPC across the first 15 demo tweets",
      labels={"method": "Method", "AOPC": "Mean probability drop"}).show()

In [ ]:
run_checks("Q12")

**Q12(d) — Answer:** The printed rankings above give the measured faithfulness order (mean AOPC, higher is better) and attribution-generation speed order (mean seconds, lower is faster) for this run. For a real-time app, choose the fastest method whose AOPC is still acceptable; for an audit report, prioritize the strongest deletion result and include the explanation itself, model confidence, and method limitations. Treat small differences cautiously because this benchmark uses only 15 tweets and a simple word-deletion perturbation.

---
# Part F — Reflection *(5 marks)*
Answer in 2–3 sentences each.

**F1.** LIME and SHAP are *model-agnostic*; IG is not. Give one advantage and one disadvantage of each kind.

**F1 — Answer:** LIME and SHAP are model-agnostic, so they can explain a black-box classifier without needing access to its gradients; however, perturbation-based explanations can be expensive and may create unrealistic text. Integrated Gradients uses model gradients and provides a completeness check relative to a baseline, but it requires differentiable model access and its results can change with the baseline choice.

**F2.** Two methods disagree on the most important word. Does that mean one of them is wrong? How would you decide which to trust?

**F2 — Answer:** Disagreement does not automatically mean one method is wrong because the methods use different assumptions and explain different aspects of the prediction. I would compare stability across runs, test faithfulness with deletion or other interventions, inspect the model probabilities, and check whether the highlighted words make sense in the original context before deciding how much to trust an explanation.

**F3.** Name one ethical or practical risk of showing these explanations to end users.

**F3 — Answer:** A key risk is that users may treat a plausible-looking explanation as proof that the prediction is correct. If the model relies on spurious or sensitive word associations, presenting those associations without uncertainty or context can mislead users and reinforce unfair decisions.

---
## 🏁 Final scorecard (given)
Run this before submitting — every automatic check in one place. *(Written answers are graded manually.)*

In [ ]:
scorecard()

### 🧾 Marking scheme
| Part | Questions | Marks |
|---|---|---|
| A — Data & Model | Q1 (5), Q2 (6 + 4 written) | 15 |
| B — LIME | Q3 (8), Q4 (6), Q5 (6) | 20 |
| C — SHAP | Q6 (8), Q7 (6), Q8 (6) | 20 |
| D — Integrated Gradients | Q9 (12), Q10 (8) | 20 |
| E — Comparison & Faithfulness | Q11 (8), Q12 (12) | 20 |
| F — Reflection | F1–F3 | 5 |
| **Total** | | **100** |

*Code tasks: 60% passing checks, 20% correct plots/outputs, 20% code clarity. Written answers must quote your own results.*

**References:** Saravia et al. (2018) *CARER: Contextualized Affect Representations for Emotion Recognition* (dataset) · Ribeiro et al. (2016) LIME · Lundberg & Lee (2017) SHAP · Sundararajan et al. (2017) Integrated Gradients.